### Installation

In [1]:
#%%capture
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if "COLAB_" not in "".join(os.environ.keys()):
    pass#!pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    pass#!pip install --no-deps unsloth vllm==0.8.5.post1

In [2]:
#@title Colab Extra Install { display-mode: "form" }
#%%capture
import os, sys, re, requests; modules = list(sys.modules.keys())
if "COLAB_" not in "".join(os.environ.keys()):
    pass#!pip install unsloth vllm
else:
    pass#!pip install --no-deps unsloth vllm==0.8.5.post1
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    #for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    #!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    #!pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    #f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    #with open("vllm_requirements.txt", "wb") as file:
    #    file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    #!pip install -r vllm_requirements.txt

Goal: To convert `Qwen3-4B-GRPO-finetune` into a multi task model via SFFT by using distilled dataset from Qwen2.5-coder-32B.

# --- 1. MODEL CONFIGURATION ---

# Set the name of the model you are starting with.
# This should be your existing fine-tuned model that is good at math reasoning.

In [3]:
import os
import pandas as pd
from datasets import Dataset, load_dataset
#from trl import SFTTrainer, SFTConfig
#import torch
import json

# --- 1. Model Configuration and initialization ---

In [4]:
#--- 1. CONFIGURATION ---
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
MAX_SEQ_LENGTH = max_seq_length
lora_rank = 32 # Larger rank = smarter, but slower

# --- 2. MODEL INITIALIZATION ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/local/project/models--unsloth--Qwen3-4B-Base/snapshots",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-08-16 15:19:07.154621: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-16 15:19:07.180758: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-16 15:19:07.180782: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-16 15:19:07.181714: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-16 15:19:07.186727: I tensorflow/core/platform/cpu_feature_guar

🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 08-16 15:19:09 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 08-16 15:19:09 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.6.6: Fast Qwen3 patching. Transformers: 4.52.4. vLLM: 0.8.5.post1.
   \\   /|    NVIDIA GeForce RTX 4090 Laptop GPU. Num GPUs = 1. Max memory: 15.992 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading /local/project/models--unsloth--Qwen3-4B-Base/snapshots with actual GPU utilization = 64.19%
Unsloth: Your GPU has CUDA compute capability 8.9 with VRAM = 15.99 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequence

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 08-16 15:26:15 [loader.py:458] Loading weights took 54.06 seconds
INFO 08-16 15:26:15 [punica_selector.py:18] Using PunicaWrapperGPU.
INFO 08-16 15:26:15 [gpu_model_runner.py:1347] Model loading took 7.6334 GiB and 55.054465 seconds
INFO 08-16 15:26:24 [backends.py:420] Using cache directory: /local/user/.cache/vllm/torch_compile_cache/f74501a6ed/rank_0_0 for vLLM's torch.compile
INFO 08-16 15:26:24 [backends.py:430] Dynamo bytecode transform time: 8.74 s
INFO 08-16 15:26:30 [backends.py:118] Directly load the compiled graph(s) for shape None from the cache, took 5.380 s
INFO 08-16 15:26:48 [monitor.py:33] torch.compile takes 8.74 s in total
INFO 08-16 15:26:48 [kv_cache_utils.py:634] GPU KV cache size: 2,416 tokens
INFO 08-16 15:26:48 [kv_cache_utils.py:637] Maximum concurrency for 2,048 tokens per request: 1.18x
INFO 08-16 15:27:23 [gpu_model_runner.py:1686] Graph capturing finished in 34 secs, took 0.83 GiB
INFO 08-16 15:27:23 [core.py:159] init engine (profile, create kv cache

Unsloth 2025.6.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


# --- 2. System Prompt ---

In [5]:
# --- 2. SYSTEM PROMPTS (Must match training) ---
code_system_prompt = \
"""You are an expert Python programmer. Your sole task is to write a self-contained Python script to solve the given computational problem.
- Your response MUST begin directly with the code block ```python and end with ```.
- Do NOT provide any text or explanation before or after the code block.
- The script must define a function `solve()` that returns the final numerical answer.
- The script must then call the `solve()` function. The result should be the final expression of the script.
- Do NOT solve the problem yourself or provide any reasoning inside the script, just return the caculation.
- Analyze the problem carefully and choose the appropriate response format, which should contain non-repetitive answers.
- For example, for '$Calculate# the value of (1+1)', your entire response must be: '```python
def solve():
    return 1+1
print(solve())
```'.
"""

cot_code_system_prompt = \
"""You are a multi-talented expert in mathematics and Python programming. Your task is to solve the given problem by providing both a textual explanation and a Python script.
- First, provide a clear, step-by-step explanation of your reasoning.
- After the explanation, provide a complete, self-contained Python script inside a ```python ... ``` block.
- The script should define a function `solve()` that returns the final numerical answer.
- The script must then call the `solve()` function. The result should be the final expression of the script.
- For example, for '$Calculate# the value of (1+1)', your script need to be: '```python
def solve():
    return 1+1
print(solve())
```'.
"""

# --- 3. DATASET PREPARATION ---

In [6]:
print("\n--- Starting 2-Category Data Preparation ---")

DISTILLED_JSONL_PATH = r"/local/project/baseline/results_v3.jsonl"
SYNTHETIC_JSONL_PATH = r"/local/project/arithmetic_logic_data_revised_v2.jsonl"

# Load data
try:
    with open(DISTILLED_JSONL_PATH, 'r', encoding='utf-8') as f:
        distilled_records = [json.loads(line) for line in f]
    print(f"Loaded {len(distilled_records)} records from {DISTILLED_JSONL_PATH}")
except FileNotFoundError:
    print(f"FATAL: Distilled data not found at {DISTILLED_JSONL_PATH}.")
    sys.exit(1)

try:
    with open(SYNTHETIC_JSONL_PATH, 'r', encoding='utf-8') as f:
        synthetic_records = [json.loads(line) for line in f]
    print(f"Loaded {len(synthetic_records)} records from {SYNTHETIC_JSONL_PATH}")
except FileNotFoundError:
    print(f"FATAL: Synthetic data not found at {SYNTHETIC_JSONL_PATH}.")
    sys.exit(1)

# Process into two pools
arithmetic_pool = []
for record in synthetic_records:
    arithmetic_pool.append({
        'problem': record.get('problem'),
        'solution': record.get('generated_python_code', ''),
        'category': 'arithmetic_only'
    })

cot_code_pool = []
for record in distilled_records:
    if record.get('is_correct') and record.get('generated_solution') and record.get('generated_python_code'):
        cot_code_pool.append({
            'problem': record.get('problem'),
            'solution_text': record.get('generated_solution', ''),
            'solution_code': record.get('generated_python_code', ''),
            'category': 'cot_and_code'
        })

print(f"\nInitial pool sizes: {len(arithmetic_pool)} Arithmetic, {len(cot_code_pool)} CoT+Code.")


--- Starting 2-Category Data Preparation ---
Loaded 1125 records from /local/project/baseline/results_v3.jsonl
Loaded 309 records from /local/project/arithmetic_logic_data_revised_v2.jsonl

Initial pool sizes: 309 Arithmetic, 251 CoT+Code.


In [7]:
def format_and_filter(pool, category, max_len):
    """Applies the correct prompt and filters by token length."""
    if not pool: return []
    
    formatted_samples = []
    for sample in pool:
        if category == 'arithmetic_only':
            system_prompt = code_system_prompt
            assistant_content = f"```python\n{sample['solution']}\n```"
        else: # cot_and_code
            system_prompt = cot_code_system_prompt
            assistant_content = f"{sample['solution_text']}\n\n```python\n{sample['solution_code']}\n```"
        
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": sample["problem"]},
            {"role": "assistant", "content": assistant_content},
        ]
        
        token_length = len(tokenizer.apply_chat_template(messages, tokenize=True))
        
        if token_length <= max_len:
            sample['messages'] = messages
            formatted_samples.append(sample)
            
    print(f"Filtered '{category}' pool: {len(formatted_samples)} samples remain out of {len(pool)}.")
    return formatted_samples

In [8]:
# Format and filter each pool
MAX_SEQ_LENGTH = max_seq_length
arithmetic_pool = format_and_filter(arithmetic_pool, 'arithmetic_only', MAX_SEQ_LENGTH)
cot_code_pool = format_and_filter(cot_code_pool, 'cot_and_code', MAX_SEQ_LENGTH)

# Combine the clean pools into the final dataset
final_samples = arithmetic_pool + cot_code_pool
final_df = pd.DataFrame(final_samples)

print(f"\nFinal dataset size for training: {len(final_df)} samples.")
print("Final data distribution:\n", final_df['category'].value_counts())

# Create the final dataset for the trainer
dataset = Dataset.from_pandas(final_df)
final_dataset = dataset.shuffle(seed=42)
final_dataset = final_dataset.map(
    lambda x: {"text": tokenizer.apply_chat_template(x["messages"], tokenize=False, add_generation_prompt=False)}
)
final_dataset = final_dataset.remove_columns([col for col in final_dataset.column_names if col != 'text'])

Filtered 'arithmetic_only' pool: 309 samples remain out of 309.
Filtered 'cot_and_code' pool: 143 samples remain out of 251.

Final dataset size for training: 452 samples.
Final data distribution:
 category
arithmetic_only    309
cot_and_code       143
Name: count, dtype: int64


Map:   0%|          | 0/452 [00:00<?, ? examples/s]

In [9]:
# To check the raw data formatted with prompt format
final_dataset["text"][0]

"You are a multi-talented expert in mathematics and Python programming. Your task is to solve the given problem by providing both a textual explanation and a Python script.\n- First, provide a clear, step-by-step explanation of your reasoning.\n- After the explanation, provide a complete, self-contained Python script inside a ```python ... ``` block.\n- The script should define a function `solve()` that returns the final numerical answer.\n- The script must then call the `solve()` function. The result should be the final expression of the script.\n- For example, for 'caculate the value of (1+1)', your script need to be: '```python\ndef solve():\n    return 1+1\nprint(solve())\n```'.\n<|endoftext|>Find the exact value of \\( \\frac{2x + 3y}{5x + 7y} \\) if \\( \\frac{y}{x} = 3 \\).<think>\nOkay, let's see. I need to find the exact value of the expression (2x + 3y)/(5x + 7y) given that y/x is 3. Hmm, so maybe I can express everything in terms of one variable, either x or y. Since the rat

In [10]:
# Apply the chat template and tell it NOT to tokenize, so we get the string back
# All the tokens will be the input of trainer, not more than 2048 in length
# To check the inupt text with chat-template info
tokenizer.apply_chat_template(final_dataset["text"][0], tokenize=False)

'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_working_out> and <end_working_out>.\nThen, provide your solution between <SOLUTION></SOLUTION><|endoftext|>'

# --- 3. SFTTRAINER CONFIGURATION ---

In [12]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = final_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8, # Use GA to mimic batch size!
        max_grad_norm=1.0,                # ADD THIS for stability
        warmup_steps = 10,
        num_train_epochs = 2, # Set this for 1 full training run.
        learning_rate = 3e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=32):   0%|          | 0/452 [00:00<?, ? examples/s]

# --- 4. START TRAINING ---

In [13]:
print("Starting model fine-tuning...")
trainer.train()
print("Fine-tuning complete.")

Starting model fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 452 | Num Epochs = 2 | Total steps = 114
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288/4,088,528,384 (1.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!


<IPython.core.display.HTML object>

Fine-tuning complete.


# --- 5. SAVE THE TRAINED LoRA ADAPTERS ---

In [14]:
output_lora_path = "/local/project/outputs/sftt_save_lora_v3.8"

print(f"Saving LoRA adapters to {output_lora_path}...")
#model.save_pretrained(output_lora_path)
model.save_lora(output_lora_path)
print("LoRA adapters saved successfully.")

# You can also save the tokenizer if you made any changes to  it, which is good practice.
tokenizer.save_pretrained(output_lora_path)
print("Tokenizer saved successfully.")

Saving LoRA adapters to /local/project/outputs/sftt_save_lora_v3.8...
LoRA adapters saved successfully.
Tokenizer saved successfully.


In [14]:
#model.save_lora("grpo_saved_lora")
#model.save_lora("/mnt/ollama_data/ubuntu/projects/2025_0627/outputs/grpo_saved_lora/lora_test")

In [20]:
import torch
from numba import cuda
torch.cuda.empty_cache()
if torch.cuda.is_available():
    print("Releasing GPU memory (cleanup)...")
    cuda.get_current_device().reset()
    print("GPU memory released.")

import gc
gc.collect()

Releasing GPU memory (cleanup)...
GPU memory released.


198

In [ ]:
# To kill the subprocess of PID directly to empty the memory of GPU in case of failure after restarting kernel 
#!kill -9 9190

In [1]:
!nvidia-smi

Wed Aug 13 14:52:02 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.28.03              Driver Version: 560.76         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090 ...    On  |   00000000:01:00.0 Off |                  N/A |
| N/A   46C    P8              9W /  136W |       0MiB /  16376MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----